**PLANT CAPTION TEXT CLASSIFIER**

Imports

In [ ]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import matplotlib.pyplot as plt

GPU Check

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


Load Dataset

In [ ]:
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Drop the image column

In [ ]:
df = df.drop(columns=["image"])
print(f"Data shape before explode: {df.shape}")

Data shape before explode: (20638, 2)


Explode captions list

In [ ]:
df = df.explode("captions")
print(f"Data shape after explode: {df.shape}")

Data shape after explode: (82552, 2)


Clean NaN values

In [ ]:
df = df.dropna(subset=["captions"])
print(f"📊 Final data shape: {df.shape}")

📊 Final data shape: (82552, 2)


Encode labels

In [ ]:
label_col = "caption"
text_col = "captions"

encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df[label_col].tolist())

num_labels = len(encoder.classes_)
print(f"Number of unique labels: {num_labels}")

Number of unique labels: 15


Train-test split

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
print(f"✅ Train shape: {df_train.shape}, Test shape: {df_test.shape}")

✅ Train shape: (66041, 3), Test shape: (16511, 3)


Convert to Hugging Face Datasets

In [ ]:
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

Load Tokenizer

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(data):
    return tokenizer(data[text_col], truncation=True, padding=False)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

Data Collator

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluation Metrics

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
        "precision": precision.compute(predictions=preds, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="weighted")["recall"]
    }


Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_strategy="epoch",
    report_to="none",
)

Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/tmp/ipython-input-2922868714.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Train Model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.230000,0.000021,1.000000,1.000000,1.000000,1.000000
2,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
3,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
4,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=33024, training_loss=0.05750048064554711, metrics={'train_runtime': 1504.5643, 'train_samples_per_second': 175.575, 'train_steps_per_second': 21.949, 'total_flos': 1844210907407070.0, 'train_loss': 0.05750048064554711, 'epoch': 4.0})

Save the Fine-Tuned Model


In [ ]:
model.save_pretrained("./plant_text_classifier")
tokenizer.save_pretrained("./plant_text_classifier")

print("Model training complete and saved!")

Model training complete and saved!


Inference Example

In [ ]:
text = "Tomato leaf showing white spots"
inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**inputs)
pred = torch.argmax(outputs.logits, dim=1).item()
predicted_label = encoder.inverse_transform([pred])[0]

print(f"Input: {text}")
print(f"Predicted Label: {predicted_label}")


Input: Tomato leaf showing white spots
Predicted Label: Tomato Leaf Mold
